# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results:
```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  16.27
Request throughput (req/s):              4.92
Output token throughput (tok/s):         1258.65
Total token throughput (tok/s):          1632.19
Mean TTFT (ms):                          78.17
Mean TPOT (ms):                          5.76
Acceptance rate:                         22.48%
Draft tokens:                            2
Acceptance length:                       1.45
Drafts:                                  14088
Draft tokens:                            28176
Accepted tokens:                         6334
==================================================
```

This run uses two speculative draft tokens. The output throughput is above the 1250 tok/s scoring threshold, and TPOT improves from the baseline 7.28 ms to 5.76 ms.


FP8 quantization benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  13.06
Request throughput (req/s):              6.12
Output token throughput (tok/s):         1566.56
Total token throughput (tok/s):          2031.82
Mean TTFT (ms):                          51.18
Mean TPOT (ms):                          4.90
Acceptance rate:                         N/A
==================================================
```

The FP8 dynamic verifier is faster than the BF16 baseline because FP8 reduces linear-layer bandwidth and activation compute pressure on H100. The result is above the 1550 tok/s scoring threshold.


FP8 + speculative decoding benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  11.59
Request throughput (req/s):              6.90
Output token throughput (tok/s):         1766.55
Total token throughput (tok/s):          2290.82
Mean TTFT (ms):                          30.24
Mean TPOT (ms):                          4.28
Acceptance rate:                         36.50%
Draft tokens:                            1
Acceptance length:                       1.36
Drafts:                                  14954
Draft tokens:                            14954
Accepted tokens:                         5458
==================================================
```

The combined FP8 + speculative configuration is the fastest run and is above the 1750 tok/s scoring threshold. The best FP8 speculative setting uses one draft token, not two, because the FP8 verifier is already cheaper and extra draft work must be accepted often enough to reduce TPOT.


## Final Answers

### Which optimization should be done first?

Train the EAGLE-3 speculative decoding draft head first, then quantize the verifier. The draft head learns from verifier hidden states and token targets, so it should see the original BF16 verifier distribution during training. FP8 dynamic quantization is then applied as a serving optimization and validated by acceptance rate, acceptance length, throughput, and TPOT.

### Task 1: Hidden-state storage

Hidden states require much more disk space than the text dataset because text is stored as compact strings or token IDs, while hidden-state generation stores dense floating-point tensors. Storage scales approximately as `sequence_length * captured_layers * hidden_size * dtype_bytes`, so thousands of samples with several BF16 hidden-state layers can become hundreds of GB.

### Task 2: Training metrics

`full_acc` measures whether drafted tokens remain correct through a speculative position. `cond_acc` measures correctness at a position conditioned on earlier draft positions already being correct. Accuracy decreases for later positions because the draft head is predicting farther ahead with less verifier computation and earlier mistakes compound. If first-position accuracy is very low, inspect data generation first: model/tokenizer match, assistant mask, hidden-state length versus token length, vLLM version, and stale `/tmp/hidden_states` files.

### Task 3: Quantization

FP8 dynamic quantization is useful on H100 because Hopper GPUs accelerate FP8 operations and the lower precision reduces memory bandwidth pressure. `lm_head` is kept unquantized because it directly controls final vocabulary logits, so quantization noise there can harm generation quality and speculative acceptance. Quantization can change verifier logits slightly, so the BF16-trained draft head may agree with the FP8 verifier at a different rate; this is why FP8 + speculative decoding is benchmarked separately.

### Task 4: Speculative benchmark interpretation

Speculative decoding improves throughput even without 100% acceptance because accepted draft tokens let one verifier step validate and emit more than one output token. For the reference setup, two draft tokens are best for the BF16 verifier speculative run, with acceptance length 1.45 and TPOT 5.76 ms. One draft token is best for FP8 + speculative decoding, with acceptance length 1.36 and TPOT 4.28 ms, because the FP8 verifier is already faster and extra draft tokens add work unless accepted often enough.
